# 🔥 FireServe GPU Backend — Colab Deployment

This notebook runs the GPU inference backend on Colab's free T4 GPU.
The gateway routes image generation requests here for actual GPU-accelerated inference.

**Setup:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order
3. Copy the public URL and register it with your gateway

In [ ]:
# Step 1: Install dependencies
!pip install -q fastapi uvicorn diffusers transformers accelerate torch pyngrok

In [ ]:
# Step 2: Verify GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Step 3: Write the GPU backend server
%%writefile server.py
from fastapi import FastAPI
from pydantic import BaseModel
import torch, base64, io, time

app = FastAPI(title="FireServe GPU Backend")
pipe = None
device = None

class GenReq(BaseModel):
    prompt: str
    negative_prompt: str = ""
    width: int = 512
    height: int = 512
    num_inference_steps: int = 4
    guidance_scale: float = 0.0
    seed: int = 42

@app.on_event("startup")
async def startup():
    global pipe, device
    from diffusers import AutoPipelineForText2Image
    device = "cuda"
    pipe = AutoPipelineForText2Image.from_pretrained(
        "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
    ).to(device)
    _ = pipe("warmup", num_inference_steps=1, width=256, height=256)
    print("Model ready!")

@app.get("/health")
async def health():
    return {
        "status": "healthy" if pipe else "loading",
        "gpu": torch.cuda.get_device_name(0),
        "vram_used_gb": round(torch.cuda.memory_allocated(0)/1e9, 2),
    }

@app.post("/generate")
async def generate(req: GenReq):
    gen = torch.Generator(device=device).manual_seed(req.seed)
    start = time.monotonic()
    image = pipe(
        prompt=req.prompt, negative_prompt=req.negative_prompt or None,
        width=req.width, height=req.height,
        num_inference_steps=req.num_inference_steps,
        guidance_scale=req.guidance_scale, generator=gen,
    ).images[0]
    ms = (time.monotonic() - start) * 1000
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return {
        "image_base64": base64.b64encode(buf.getvalue()).decode(),
        "inference_time_ms": round(ms, 2),
        "gpu_type": torch.cuda.get_device_name(0),
    }

In [ ]:
# Step 4: Start server with ngrok tunnel
import subprocess, threading, time
from pyngrok import ngrok

# Start uvicorn in background
proc = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8001"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
time.sleep(5)  # Wait for model to load

# Create public URL
public_url = ngrok.connect(8001)
print(f"\n{'='*60}")
print(f"GPU Backend is LIVE!")
print(f"Public URL: {public_url}")
print(f"Health:     {public_url}/health")
print(f"{'='*60}")
print(f"\nRegister with gateway:")
print(f'  curl -X POST "http://localhost:8000/backends/register?url={public_url}&gpu_type=T4&vram_gb=15"')

In [ ]:
# Step 5: Test it locally
import requests, base64
from PIL import Image
from IPython.display import display

resp = requests.post(f"{public_url}/generate", json={
    "prompt": "a majestic mountain landscape at golden hour, digital art",
    "num_inference_steps": 4,
    "width": 512, "height": 512,
})
data = resp.json()
print(f"Inference time: {data['inference_time_ms']:.0f}ms on {data['gpu_type']}")

img_bytes = base64.b64decode(data["image_base64"])
img = Image.open(io.BytesIO(img_bytes))
display(img)